In [1]:
import json
import pandas as pd
from pathlib import Path

pred_root = Path("/scratch/jq2uw/MME/instruct_vlm_edit/results/pred")

# VLM name -> folder name
vlms = {
    "qwen3":    "Qwen3-VL-8B-Instruct",
    "qwen3_4b": "Qwen3-VL-4B-Instruct",
    "llava":    "llava-1.5-7b-hf",
    # "blip":     "instructblip-vicuna-7b",
}

# Load each VLM's predictions
preds = {}
for short, folder in vlms.items():
    path = pred_root / folder / "midas" / "mc_all.json"
    if path.exists():
        with open(path) as f:
            preds[short] = json.load(f)
        print(f"Loaded {short}: {len(preds[short])} examples")
    else:
        print(f"Skipped {short}: {path} not found")

# Build base table from first available VLM
first = preds[next(iter(preds))]
df = pd.DataFrame([{
    "uid": ex["uid"],
    "image": ex["image"],
    "question": ex["question"],
    "answer": ex["answer"],
} for ex in first])

# Add per-VLM columns
for short, data in preds.items():
    lookup = {ex["uid"]: ex.get("pred", {}) for ex in data}
    df[f"{short}_answer"]  = df["uid"].map(lambda u, lk=lookup: lk.get(u, {}).get("answer", ""))
    df[f"{short}_maxprob"] = df["uid"].map(lambda u, lk=lookup: lk.get(u, {}).get("label_maxprob", ""))

print(f"\nFinal table: {len(df)} rows x {len(df.columns)} cols")
df.head()

Loaded qwen3: 3357 examples
Loaded qwen3_4b: 3357 examples
Loaded llava: 3357 examples

Final table: 3357 rows x 10 cols


,uid,image,question,answer,qwen3_answer,qwen3_maxprob,qwen3_4b_answer,qwen3_4b_maxprob,llava_answer,llava_maxprob
0,1,data/images/midas/s-prd-398966407.jpg,"Is the lesion malignant, benign, or other? Pro...",malignant,"Based on the image provided, the lesion appear...",benign,"Based on the image provided, the lesion appear...",other,"The lesion is described as a small, pink, and ...",benign
1,2,data/images/midas/s-prd-398966642.jpg,"Is the lesion malignant, benign, or other? Pro...",malignant,"Based on the image provided, the lesion appear...",benign,"Based on the image provided, the lesion appear...",benign,"The lesion is described as a small, red, and p...",benign
2,3,data/images/midas/s-prd-398966845.jpg,"Is the lesion malignant, benign, or other? Pro...",malignant,"Based on the provided dermoscopy image, the le...",benign,"Based on the dermoscopic image provided, the l...",benign,"The lesion is described as a malignant lesion,...",benign
3,4,data/images/midas/s-prd-398967381.jpg,"Is the lesion malignant, benign, or other? Pro...",benign,"Based on the image provided, it is **not possi...",benign,"Based on the image provided, it is **not possi...",unknown,"The lesion is malignant, as it is described as...",malignant
4,5,data/images/midas/s-prd-398967587.jpg,"Is the lesion malignant, benign, or other? Pro...",benign,"Based on the provided image, the lesion appear...",benign,BenignRationaleThis lesion is a well-circumscr...,benign,"The lesion is described as a small, pink, and ...",malignant


In [2]:
# Merge SkinGPT-4 predictions
skingpt_path = pred_root / "skingpt4" / "raw_skingpt4_predictions.csv"
skingpt_df = pd.read_csv(skingpt_path)
skingpt_df = skingpt_df[["uid", "response", "pred_label"]].rename(columns={
    "response":   "skingpt_answer",
    "pred_label":  "skingpt_maxprob",
})
skingpt_df["uid"] = skingpt_df["uid"].astype(int)
df["uid"] = df["uid"].astype(int)

# Remove ###NLL:{...} suffix from skingpt answers
skingpt_df["skingpt_answer"] = skingpt_df["skingpt_answer"].str.replace(r"\n###NLL:.*", "", regex=True)

df = df.merge(skingpt_df, on="uid", how="left")

# Normalize "unknown" -> "other" across all maxprob columns
for c in df.columns:
    if c.endswith("_maxprob"):
        df[c] = df[c].replace("unknown", "other")
print(f"After SkinGPT-4 merge: {len(df)} rows x {len(df.columns)} cols")
df.head()

After SkinGPT-4 merge: 3357 rows x 12 cols


,uid,image,question,answer,qwen3_answer,qwen3_maxprob,qwen3_4b_answer,qwen3_4b_maxprob,llava_answer,llava_maxprob,skingpt_answer,skingpt_maxprob
0,1,data/images/midas/s-prd-398966407.jpg,"Is the lesion malignant, benign, or other? Pro...",malignant,"Based on the image provided, the lesion appear...",benign,"Based on the image provided, the lesion appear...",other,"The lesion is described as a small, pink, and ...",benign,"This is an image of a person's arm, showing re...",other
1,2,data/images/midas/s-prd-398966642.jpg,"Is the lesion malignant, benign, or other? Pro...",malignant,"Based on the image provided, the lesion appear...",benign,"Based on the image provided, the lesion appear...",benign,"The lesion is described as a small, red, and p...",benign,"The image shows a large, red and swollen mole ...",other
2,3,data/images/midas/s-prd-398966845.jpg,"Is the lesion malignant, benign, or other? Pro...",malignant,"Based on the provided dermoscopy image, the le...",benign,"Based on the dermoscopic image provided, the l...",benign,"The lesion is described as a malignant lesion,...",benign,This image appears to be a close up view of a ...,malignant
3,4,data/images/midas/s-prd-398967381.jpg,"Is the lesion malignant, benign, or other? Pro...",benign,"Based on the image provided, it is **not possi...",benign,"Based on the image provided, it is **not possi...",other,"The lesion is malignant, as it is described as...",malignant,This image appears to be a scan of a woman's a...,other
4,5,data/images/midas/s-prd-398967587.jpg,"Is the lesion malignant, benign, or other? Pro...",benign,"Based on the provided image, the lesion appear...",benign,BenignRationaleThis lesion is a well-circumscr...,benign,"The lesion is described as a small, pink, and ...",malignant,This image shows a close up view of a woman's ...,other


In [3]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

labels = ["malignant", "benign", "other"]
models = ["qwen3", "qwen3_4b", "llava", "skingpt"]

for model in models:
    col = f"{model}_maxprob"
    if col not in df.columns:
        print(f"--- {model}: column '{col}' not found, skipping ---\n")
        continue

    # Drop rows where prediction is missing/NaN
    mask = df[col].notna() & (df[col] != "")
    y_true = df.loc[mask, "answer"]
    y_pred = df.loc[mask, col]

    # Build confusion matrix with fixed label order
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_df = pd.DataFrame(cm, index=labels, columns=labels)
    cm_df.index.name = "true \\ pred"

    print(f"=== {model} ===")
    print(f"Rows evaluated: {mask.sum()} / {len(df)}")
    # Show values not in standard labels
    unexpected = set(y_pred.unique()) - set(labels)
    if unexpected:
        print(f"Non-standard predictions dropped from matrix: {unexpected}")
    print(cm_df)
    print()
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))
    print("-" * 60, "\n")

=== qwen3 ===
Rows evaluated: 3357 / 3357
             malignant  benign  other
true \ pred                          
malignant           70    1246     75
benign              13    1277     32
other                1     618     25

              precision    recall  f1-score   support

   malignant       0.83      0.05      0.09      1391
      benign       0.41      0.97      0.57      1322
       other       0.19      0.04      0.06       644

    accuracy                           0.41      3357
   macro avg       0.48      0.35      0.24      3357
weighted avg       0.54      0.41      0.28      3357

------------------------------------------------------------ 

=== qwen3_4b ===
Rows evaluated: 3357 / 3357
             malignant  benign  other
true \ pred                          
malignant           24    1248    119
benign               6    1219     97
other                2     588     54

              precision    recall  f1-score   support

   malignant       0.75      0.0

In [4]:
from sklearn.metrics import confusion_matrix as cm_func, accuracy_score, balanced_accuracy_score

labels = ["malignant", "benign", "other"]
models = ["qwen3", "qwen3_4b", "llava", "skingpt"]

# Build performance table: confusion matrix rows + accuracy row per model
perf_rows = []
for model in models:
    col = f"{model}_maxprob"
    if col not in df.columns:
        continue
    mask = df[col].notna() & (df[col] != "")
    y_true = df.loc[mask, "answer"]
    y_pred = df.loc[mask, col]

    cm = cm_func(y_true, y_pred, labels=labels)
    for i, true_label in enumerate(labels):
        row = {"model": model, "metric": f"{true_label} (true)"}
        for j, pred_label in enumerate(labels):
            row[pred_label] = cm[i, j]
        row["accuracy"] = ""
        row["balanced_accuracy"] = ""
        perf_rows.append(row)

    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    perf_rows.append({
        "model": model, "metric": "accuracy",
        "malignant": "", "benign": "", "other": "",
        "accuracy": round(acc, 4),
        "balanced_accuracy": round(bal_acc, 4),
    })

perf_df = pd.DataFrame(perf_rows, columns=["model", "metric", "malignant", "benign", "other", "accuracy", "balanced_accuracy"])

# Save to Excel with two sheets
out_path = Path("midas/vlm_generation.xlsx")
out_path.parent.mkdir(parents=True, exist_ok=True)
# Column dictionary for the generation sheet
dict_df = pd.DataFrame({
    "column": [
        "uid", "image", "question", "answer",
        "qwen3_answer", "qwen3_maxprob",
        "qwen3_4b_answer", "qwen3_4b_maxprob",
        "llava_answer", "llava_maxprob",
        "skingpt_answer", "skingpt_maxprob",
    ],
    "definition": [
        "Unique ID",
        "Image file path",
        "Prompt given to VLM",
        "Ground truth label",
        "Qwen3-8B free-text response",
        "Qwen3-8B predicted label by softmax probability, from three labels: malignant, benign, other",
        "Qwen3-4B free-text response",
        "Qwen3-4B predicted label by softmax probability, from three labels: malignant, benign, other",
        "LLaVA-1.5-7B free-text response",
        "LLaVA-1.5-7B predicted label by softmax probability, from three labels: malignant, benign, other",
        "SkinGPT free-text response",
        "SkinGPT predicted label by softmax probability, from three labels: malignant, benign, other",
    ],
})

with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    dict_df.to_excel(writer, sheet_name="dictionary", index=False)
    df.to_excel(writer, sheet_name="generation", index=False)
    perf_df.to_excel(writer, sheet_name="performance", index=False)

print(f"Saved to {out_path}")
print(f"  generation:  {len(df)} rows")
print(f"  performance: {len(perf_df)} rows")
perf_df

Saved to midas/vlm_generation.xlsx
  generation:  3357 rows
  performance: 16 rows


,model,metric,malignant,benign,other,accuracy,balanced_accuracy
0,qwen3,malignant (true),70,1246,75,,
1,qwen3,benign (true),13,1277,32,,
2,qwen3,other (true),1,618,25,,
3,qwen3,accuracy,,,,0.4087,0.3517
4,qwen3_4b,malignant (true),24,1248,119,,
5,qwen3_4b,benign (true),6,1219,97,,
6,qwen3_4b,other (true),2,588,54,,
7,qwen3_4b,accuracy,,,,0.3864,0.3411
8,llava,malignant (true),663,727,1,,
9,llava,benign (true),659,663,0,,
